In [8]:
from scapy.all import rdpcap, Raw, UDP
OPENVPN_PORT = 1194
def extract_opcode(byte):
    return (byte & 0b11111000) >> 3
HARD_RESET = {1, 2, 7, 8, 10}  
CONTROL = {4}
ACK = {5}
DATA = {6, 9}

def is_likely_openvpn(pkt):
    return (
        UDP in pkt and Raw in pkt and
        (pkt[UDP].sport == OPENVPN_PORT or pkt[UDP].dport == OPENVPN_PORT) and
        len(pkt[Raw].load) >= 1
    )

def analyze_openvpn_pattern(pcap_file):
    packets = rdpcap(pcap_file)
    opcodes = []

    for pkt in packets:
        if is_likely_openvpn(pkt):
            byte = pkt[Raw].load[0]
            opcode = extract_opcode(byte)
            opcodes.append(opcode)

    print(f"\n📦 Total OpenVPN-candidate packets: {len(opcodes)}")
    print(f"📚 Unique opcodes found: {set(opcodes)}")

    phase = "RESET"
    reset_seen = control_seen = ack_seen = data_seen = False

    for op in opcodes:
        if op in HARD_RESET and not reset_seen:
            reset_seen = True
            continue
        elif op in CONTROL and reset_seen and not control_seen:
            control_seen = True
            continue
        elif op in ACK and control_seen and not ack_seen:
            ack_seen = True
            continue
        elif op in DATA and ack_seen:
            data_seen = True
            break 

 
    if reset_seen and control_seen and ack_seen and data_seen:
        print("\n✅ Pattern Match: OpenVPN-like flow detected.")
        print("   Pattern: HARD RESET → CONTROL → ACK → DATA")
    else:
        print("\n❌ Pattern does NOT match standard OpenVPN handshake.")
        print("   Observed phases:")
        print(f"   RESET: {reset_seen}, CONTROL: {control_seen}, ACK: {ack_seen}, DATA: {data_seen}")


analyze_openvpn_pattern(r"D:\PR\VPN data sets\VPN data sets\43.204.236.255\vpn traffic.pcap")



📦 Total OpenVPN-candidate packets: 426
📚 Unique opcodes found: {4, 5, 8, 9, 10, 11}

✅ Pattern Match: OpenVPN-like flow detected.
   Pattern: HARD RESET → CONTROL → ACK → DATA
